<a href="https://colab.research.google.com/github/n-zad/av-vlm-belief-persistence/blob/main/notebooks/colab_run_llava7b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Belief-persistence evaluation on Colab

**Before running:**
1. *Runtime → Change runtime type → T4 GPU*.
2. Upload `data/cam_front_images.zip` (the 804 front-camera images, 118 MB) to Google Drive at `MyDrive/cmpe249/`.

Results are written to that Drive folder, and `run` skips finished sequences, so if Colab disconnects
just reconnect and run the cells again; it continues where it stopped.

In [4]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Settings

In [5]:
DRIVE_DIR = "/content/drive/MyDrive/cmpe249"   # holds cam_front_images.zip; results go here too
BRANCH = "main"

# Run once per model: Qwen first, then switch to the LLaVA line and run from step 3 again.
# MAX_SIDE = longest image side fed to the model. LLaVA-OneVision tiles each image, so a 6-frame
# history at 1024 px is ~27.7k tokens and runs out of memory on a T4; 768 px is ~14.9k.
# (Qwen at 1024 px is ~5.3k.) Keep MAX_SIDE fixed for a model once its run has started.
# MODEL, TAG, MAX_SIDE = "Qwen/Qwen2.5-VL-7B-Instruct", "qwen7b", 1024
MODEL, TAG, MAX_SIDE = "llava-hf/llava-onevision-qwen2-7b-ov-hf", "llava7b", 768

RESULTS = f"{DRIVE_DIR}/results_{TAG}.jsonl"

## 2. Setup: Drive, code, packages, images

In [6]:
from google.colab import drive
drive.mount("/content/drive")

!if [ -d /content/repo ]; then git -C /content/repo pull -q; else git clone -q -b {BRANCH} https://github.com/n-zad/av-vlm-belief-persistence.git /content/repo; fi
%cd /content/repo
!pip install -q -U transformers accelerate bitsandbytes
!mkdir -p data/nuscenes && unzip -q -o {DRIVE_DIR}/cam_front_images.zip -d data/nuscenes
!echo "images: $(ls data/nuscenes/samples/CAM_FRONT | wc -l) (expect 804)"

Mounted at /content/drive
/content/repo
images: 804 (expect 804)


## 3. Smoke test: one sequence, all three modes

Check that the raw answers are sensible JSON before the long run. If they are empty or gibberish,
stop here and report it. If you get a CUDA out-of-memory error, lower `MAX_SIDE` in step 1 (e.g. 640).

In [7]:
!rm -f /content/smoke.jsonl
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m belief_eval run --dataroot data/nuscenes --sequences sequences_trainval.jsonl --model {MODEL} --4bit --max-side {MAX_SIDE} --limit 1 --out /content/smoke.jsonl

import json
for rec in map(json.loads, open("/content/smoke.jsonl")):
    print(f"--- {rec['mode']} (evidence frame {rec['evidence_index']})")
    for f in rec["frames"]:
        print(f"t={f['t']} present={f['present']} action={f['action']} gt_action={f['gt_action']} | {f['raw'][:150]!r}")

config.json: 100% 2.53k/2.53k [00:00<00:00, 9.63MB/s]
model.safetensors.index.json: 100% 78.0k/78.0k [00:00<00:00, 172MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/4.91G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/9.84G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/16.1G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   3% 505M/16.1G [00:05<03:11, 81.2MB/s,  223MB/s  ]
Reconstructing (incomplete total...):   5% 859M/16.1G [00:15<08:02, 31.5MB/s, 63.5MB/s  ]
Reconstructing (incomplete total...):   6% 974M/16.1G [00:34<05:23, 46.6MB/s, 46.6MB/s  ]
Reconstructing (incomplete total...):  11% 1.85G/16.1G [00:57<04:34, 51.7MB/s, 17.2MB/s  ]
Reconstructing (incomplete total...):  13% 2.05G/16.1G [00:58<03:25, 68.1MB/s, 57.9MB/s  ]
Reconstructing (incomplete total...):  25% 4.09G/16.1G [03:04<09:35, 20.8MB/s, 20

## 4. Full run: 145 sequences × 3 modes

Expect several hours per model. Safe to re-run after a disconnect.
If you change `MAX_SIDE` after a partial run, delete that model's results file in Drive first,
otherwise the finished sequences keep their old image size.

In [8]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m belief_eval run --dataroot data/nuscenes --sequences sequences_trainval.jsonl --model {MODEL} --4bit --max-side {MAX_SIDE} --out {RESULTS}
!echo "finished records: $(wc -l < {RESULTS}) / 435"

Loading weights: 100% 765/765 [01:03<00:00, 12.04it/s] 
results -> /content/drive/MyDrive/cmpe249/results_llava7b.jsonl
finished records: 435 / 435


## 5. Scores (all models run so far)

In [9]:
!python -m belief_eval score {DRIVE_DIR}/results_*.jsonl

{
  "Qwen/Qwen2.5-VL-7B-Instruct": {
    "reconsider": {
      "n_sequences": 145,
      "n_hallucinated": 139,
      "persistence_rate": 0.9640287769784173,
      "persistence_rate_ci": [
        0.9280575539568345,
        0.9928057553956835
      ],
      "latency_median": 2,
      "latency_p90": 4,
      "n_never_corrected": 126,
      "post_evidence_accuracy": 0.06712643678160919,
      "post_evidence_accuracy_ci": [
        0.0367816091954023,
        0.10367816091954023
      ],
      "frame_accuracy": 0.05747126436781609,
      "safety_balanced_accuracy": 0.5335325538504367,
      "unsafe_action_rate": 0.034482758620689655
    },
    "stateful": {
      "n_sequences": 145,
      "n_hallucinated": 139,
      "persistence_rate": 0.9568345323741008,
      "persistence_rate_ci": [
        0.920863309352518,
        0.9856115107913669
      ],
      "latency_median": 2,
      "latency_p90": 4,
      "n_never_corrected": 122,
      "post_evidence_accuracy": 0.09195402298850575,
     